In [526]:
import pandas as pd
import numpy as np
from pathlib import Path

data = pd.read_csv("../data/sites_data.csv")


In [527]:
rec_risk_limit={
    "high_max": 0.70,
    "med_max": 1,
}
reliab_risk_limit = {
    "high_max": 0.70,
    "med_max": 0.85,
}


In [528]:
data["rec_risk"] = np.select(
    [
        data["rec_perf_ratio"]< rec_risk_limit["high_max"],
        data["rec_perf_ratio"]< rec_risk_limit["med_max"],
    ],
    [
        "High",  #inferior to 0.75% of reference
        "Med",#from 75-<1
    ],
    default="Low"
)
data[["site_id", "rec_perf_ratio", "rec_risk"]].head(5)

,site_id,rec_perf_ratio,rec_risk
0,Site_001,1.24,Low
1,Site_002,0.52,High
2,Site_003,0.94,Med
3,Site_004,1.01,Low
4,Site_005,0.58,High


In [529]:
data["reliab_risk"] = np.select(
    [
        data["op_reliab"] < reliab_risk_limit["high_max"],
        data["op_reliab"] < reliab_risk_limit["med_max"],
    ],
    [
        "High",
        "Med",
    ],
    default="Low",
)

In [530]:
risk_priority={
    "Low":1,
    "Med": 2,
    "High": 3.
}

In [531]:
#risk classifier
risk_columns = [
    "rec_risk",
    "reliab_risk",
]

high_count = (data[risk_columns] == "High").sum(axis=1)
med_count = data[risk_columns].isin(["Med", "Medium"]).sum(axis=1)

data["risk_level"] = np.select(
    [
        high_count >= 2,
        (high_count == 1) & (med_count == 2),
        (high_count == 1) | (med_count >= 2),
    ],
    [
        "High",
        "High",
        "Medium",
    ],
    default="Low",
)



In [532]:
#controlpoint
data[
    [
        "site_id",
        "rec_risk",
        "reliab_risk",
        "risk_level",
    ]
].head(5)

,site_id,rec_risk,reliab_risk,risk_level
0,Site_001,Low,High,Medium
1,Site_002,High,Low,Medium
2,Site_003,Med,Med,Medium
3,Site_004,Low,Med,Low
4,Site_005,High,Low,Medium


In [533]:
data["risk_level"].value_counts()

risk_level
Medium    74
Low       63
High      13
Name: count, dtype: int64

In [534]:
#validation+
print("Dataset dimensions:", data.shape)
print("Unique site IDs:", data["site_id"].nunique())
print("Missing values:", data.isna().sum().sum())
print("Duplicate site IDs:", data["site_id"].duplicated().sum())

print("\nOverall risk distribution:")
print(data["risk_level"].value_counts())


Dataset dimensions: (150, 10)
Unique site IDs: 150
Missing values: 0
Duplicate site IDs: 0

Overall risk distribution:
risk_level
Medium    74
Low       63
High      13
Name: count, dtype: int64


In [535]:
risk_columns = [ "rec_risk", "reliab_risk"]

for column in risk_columns:
    print(f"\n{column}")
    print(data[column].value_counts())
    print((data[column].value_counts(normalize=True) * 100).round(1))


rec_risk
rec_risk
Med     57
Low     54
High    39
Name: count, dtype: int64
rec_risk
Med     38.0
Low     36.0
High    26.0
Name: proportion, dtype: float64

reliab_risk
reliab_risk
Low     59
Med     51
High    40
Name: count, dtype: int64
reliab_risk
Low     39.3
Med     34.0
High    26.7
Name: proportion, dtype: float64


In [536]:
print(data.groupby("risk_level")[
    ["rec_perf_ratio", "op_reliab"]
].median().round(2))

            rec_perf_ratio  op_reliab
risk_level                           
High                  0.59       0.65
Low                   1.11       0.87
Medium                0.78       0.78


In [537]:
data.to_csv("../data/risks.csv", index=False)